In [22]:
import sys
import os
import math
import pandas as pd
sys.path.append("../")
sys.path.append("../..")

In [23]:
from scale_rl.common.wandb_utils import *

In [24]:
from scale_rl.envs.mujoco import MUJOCO_ALL, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE
from scale_rl.envs.dmc import DMC_EASY_MEDIUM, DMC_HARD
from scale_rl.envs.humanoid_bench import HB_LOCOMOTION_NOHAND, HB_RANDOM_SCORE, HB_SUCCESS_SCORE
from scale_rl.envs.myosuite import MYOSUITE_TASKS

def replace_hypen_to_underbar(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list
def replace_hypen_to_underbar_dict(env_name_dict):
    _new_dict = {}
    for k, v in env_name_dict.items():
        _new_dict[k.replace('_', '-')] = v
    return _new_dict

MUJOCO_ALL = replace_hypen_to_underbar(MUJOCO_ALL)
DMC_EASY_MEDIUM = replace_hypen_to_underbar(DMC_EASY_MEDIUM)
DMC_HARD = replace_hypen_to_underbar(DMC_HARD)
HB_LOCOMOTION_NOHAND = replace_hypen_to_underbar(HB_LOCOMOTION_NOHAND)
MYOSUITE_TASKS = replace_hypen_to_underbar(MYOSUITE_TASKS)

HB_SUCCESS_SCORE = replace_hypen_to_underbar_dict(HB_SUCCESS_SCORE)
HB_RANDOM_SCORE = replace_hypen_to_underbar_dict(HB_RANDOM_SCORE)

DMC_ALL = [*DMC_EASY_MEDIUM, *DMC_HARD]

In [43]:
from rliable import library as rly
from rliable import metrics as rly_metrics
from rliable import plot_utils as rly_plot_utils

def aggregate_mean(scores: np.ndarray):
  """Computes mean of sample mean scores per task.

  Args:
    scores: A matrix of size (`num_runs` x `num_tasks`) where scores[n][m]
      represent the score on run `n` of task `m`.
  Returns:
    Mean of sample means.
  """
  return np.mean(scores)

def aggregate_median(scores: np.ndarray):
  """Computes mean of sample mean scores per task.

  Args:
    scores: A matrix of size (`num_runs` x `num_tasks`) where scores[n][m]
      represent the score on run `n` of task `m`.
  Returns:
    Mean of sample means.
  """
  return np.median(scores)

aggregate_func = lambda x: np.array([
  rly_metrics.aggregate_iqm(x),
  aggregate_median(x),
  rly_metrics.aggregate_mean(x),])

In [44]:
MUJ_STEPS = 1000000 # 1M
DMC_STEPS = 1000000 # 1M
MYO_STEPS = 1000000 # 1M
HB_STEPS = 1000000 # 1M

In [70]:
exp_name = "PPO"
env_step = MUJ_STEPS
metric = "avg_return"

your_dict = {
    "Ant-v4": float(1584),
    "HalfCheetah-v4": float(1744),
    "Hopper-v4": float(3022),
    "Humanoid-v4": float(477),
    "Walker2d-v4": float(2487)
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)
df = normalize_score_with_random_and_base_score(
    df=df, 
    random_score_dict=MUJOCO_RANDOM_SCORE,
    base_score_dict=MUJOCO_TD3_SCORE,
)
print(df)
df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")

/tmp/ipykernel_1993396/388111719.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


  exp_name        env_name env_step      metric     value seed
0      PPO          Ant-v4  1000000  avg_return  0.412305    0
1      PPO  HalfCheetah-v4  1000000  avg_return  0.187180    0
2      PPO       Hopper-v4  1000000  avg_return  0.936393    0
3      PPO     Humanoid-v4  1000000  avg_return  0.070685    0
4      PPO     Walker2d-v4  1000000  avg_return  0.629997    0
IQM
0.41 \textcolor{gray}{[0.11, 0.834]}
Median
0.412 \textcolor{gray}{[0.071, 0.936]}
Mean
0.447 \textcolor{gray}{[0.186, 0.725]}


In [67]:
exp_name = "SAC"
env_step = MUJ_STEPS
metric = "avg_return"

your_dict = {
    "Ant-v4": float(3681),
    "HalfCheetah-v4": float(10484),
    "Hopper-v4": float(2785),
    "Humanoid-v4": float(4090),
    "Walker2d-v4": float(4314)
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)
df = normalize_score_with_random_and_base_score(
    df=df, 
    random_score_dict=MUJOCO_RANDOM_SCORE,
    base_score_dict=MUJOCO_TD3_SCORE,
)
print(df)
df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")

/tmp/ipykernel_1993396/1587849056.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


  exp_name        env_name env_step      metric     value seed
0      SAC          Ant-v4  1000000  avg_return  0.934950    0
1      SAC  HalfCheetah-v4  1000000  avg_return  0.991715    0
2      SAC       Hopper-v4  1000000  avg_return  0.862497    0
3      SAC     Humanoid-v4  1000000  avg_return  0.786900    0
4      SAC     Walker2d-v4  1000000  avg_return  1.093325    0
IQM
0.93 \textcolor{gray}{[0.812, 1.059]}
Median
0.935 \textcolor{gray}{[0.787, 1.093]}
Mean
0.934 \textcolor{gray}{[0.846, 1.027]}


In [71]:
exp_name = "TD3+OFE"
env_step = MUJ_STEPS
metric = "avg_return"

your_dict = {
    "Ant-v4": float(7398),
    "HalfCheetah-v4": float(13758),
    "Hopper-v4": float(3121),
    "Humanoid-v4": float(6032),
    "Walker2d-v4": float(5195)
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)
df = normalize_score_with_random_and_base_score(
    df=df, 
    random_score_dict=MUJOCO_RANDOM_SCORE,
    base_score_dict=MUJOCO_TD3_SCORE,
)
print(df)
df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")

/tmp/ipykernel_1993396/3562829401.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


  exp_name        env_name env_step      metric     value seed
0  TD3+OFE          Ant-v4  1000000  avg_return  1.861354    0
1  TD3+OFE  HalfCheetah-v4  1000000  avg_return  1.293094    0
2  TD3+OFE       Hopper-v4  1000000  avg_return  0.967261    0
3  TD3+OFE     Humanoid-v4  1000000  avg_return  1.171868    0
4  TD3+OFE     Walker2d-v4  1000000  avg_return  1.316747    0
IQM
1.261 \textcolor{gray}{[1.035, 1.68]}
Median
1.293 \textcolor{gray}{[0.967, 1.861]}
Mean
1.322 \textcolor{gray}{[1.09, 1.615]}


In [72]:
exp_name = "TQC"
env_step = MUJ_STEPS
metric = "avg_return"

your_dict = {
    "Ant-v4": float(3582),
    "HalfCheetah-v4": float(12349),
    "Hopper-v4": float(3526),
    "Humanoid-v4": float(6029),
    "Walker2d-v4": float(5321)
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)
df = normalize_score_with_random_and_base_score(
    df=df, 
    random_score_dict=MUJOCO_RANDOM_SCORE,
    base_score_dict=MUJOCO_TD3_SCORE,
)
print(df)
df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")

/tmp/ipykernel_1993396/108473768.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


  exp_name        env_name env_step      metric     value seed
0      TQC          Ant-v4  1000000  avg_return  0.910276    0
1      TQC  HalfCheetah-v4  1000000  avg_return  1.163392    0
2      TQC       Hopper-v4  1000000  avg_return  1.093539    0
3      TQC     Humanoid-v4  1000000  avg_return  1.171273    0
4      TQC     Walker2d-v4  1000000  avg_return  1.348701    0
IQM
1.143 \textcolor{gray}{[0.971, 1.29]}
Median
1.163 \textcolor{gray}{[0.91, 1.349]}
Mean
1.137 \textcolor{gray}{[1.012, 1.261]}


In [73]:
exp_name = "PPO"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "dog-run": float(26),
    "dog-stand": float(129),
    "dog-trot": float(31),
    "dog-walk": float(40),
    "humanoid-run": float(0),
    "humanoid-stand": float(5),
    "humanoid-walk": float(1),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)
    
df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)

aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")

/tmp/ipykernel_1993396/258544011.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


{'PPO': array([20.6       , 26.        , 33.14285714])}
IQM
20.6 \textcolor{gray}{[3.4, 69.2]}
Median
26.0 \textcolor{gray}{[1.0, 40.0]}
Mean
33.143 \textcolor{gray}{[9.139, 67.861]}


In [74]:
exp_name = "DreamerV3"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "dog-run": float(4),
    "dog-stand": float(22),
    "dog-trot": float(10),
    "dog-walk": float(18),
    "humanoid-run": float(0),
    "humanoid-stand": float(5),
    "humanoid-walk": float(1),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

/tmp/ipykernel_1993396/1694082833.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


{'DreamerV3': array([[ 4.],
       [22.],
       [10.],
       [18.],
       [ 0.],
       [ 5.],
       [ 1.]])}
IQM
7.6 \textcolor{gray}{[2.0, 15.6]}
Median
5.0 \textcolor{gray}{[1.0, 18.0]}
Mean
8.571 \textcolor{gray}{[3.143, 14.714]}


In [75]:
exp_name = "TD7"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "dog-run": float(69),
    "dog-stand": float(582),
    "dog-trot": float(21),
    "dog-walk": float(52),
    "humanoid-run": float(57),
    "humanoid-stand": float(317),
    "humanoid-walk": float(176),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

/tmp/ipykernel_1993396/2404751156.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


{'TD7': array([[ 69.],
       [582.],
       [ 21.],
       [ 52.],
       [ 57.],
       [317.],
       [176.]])}
IQM
134.2 \textcolor{gray}{[47.4, 342.8]}
Median
69.0 \textcolor{gray}{[52.0, 317.0]}
Mean
182.0 \textcolor{gray}{[62.286, 336.143]}


In [76]:
exp_name = "TD-MPC2"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "dog-run": float(265),
    "dog-stand": float(506),
    "dog-trot": float(407),
    "dog-walk": float(486),
    "humanoid-run": float(181),
    "humanoid-stand": float(658),
    "humanoid-walk": float(754),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

/tmp/ipykernel_1993396/1563908035.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


{'TD-MPC2': array([[265.],
       [506.],
       [407.],
       [486.],
       [181.],
       [658.],
       [754.]])}
IQM
464.4 \textcolor{gray}{[305.0, 631.6]}
Median
486.0 \textcolor{gray}{[265.0, 658.0]}
Mean
465.286 \textcolor{gray}{[329.425, 605.714]}


In [77]:
exp_name = "MR.Q"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "dog-run": float(569),
    "dog-stand": float(967),
    "dog-trot": float(877),
    "dog-walk": float(916),
    "humanoid-run": float(200),
    "humanoid-stand": float(868),
    "humanoid-walk": float(662),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

/tmp/ipykernel_1993396/1089523559.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


{'MR.Q': array([[569.],
       [967.],
       [877.],
       [916.],
       [200.],
       [868.],
       [662.]])}
IQM
778.4 \textcolor{gray}{[499.8, 910.6]}
Median
868.0 \textcolor{gray}{[569.0, 916.0]}
Mean
722.714 \textcolor{gray}{[516.429, 885.614]}


In [58]:
exp_name = "DreamerV3"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "acrobot-swingup": float(230),
    "ball-in-cup-catch": float(968),
    "cartpole-balance": float(998),
    "cartpole-balance-sparse": float(1000),
    "cartpole-swingup": float(736),
    "cartpole-swingup-sparse": float(702),
    "cheetah-run": float(917),
    "finger-spin": float(666),
    "finger-turn-easy": float(906),
    "finger-turn-hard": float(864),
    "fish-swim": float(813),
    "hopper-hop": float(116),
    "hopper-stand": float(747),
    "pendulum-swingup": float(774),
    "quadruped-run": float(130),
    "quadruped-walk": float(193),
    "reacher-easy": float(966),
    "reacher-hard": float(919),
    "walker-run": float(510),
    "walker-stand": float(941),
    "walker-walk": float(898),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

{'DreamerV3': array([[ 230.],
       [ 968.],
       [ 998.],
       [1000.],
       [ 736.],
       [ 702.],
       [ 917.],
       [ 666.],
       [ 906.],
       [ 864.],
       [ 813.],
       [ 116.],
       [ 747.],
       [ 774.],
       [ 130.],
       [ 193.],
       [ 966.],
       [ 919.],
       [ 510.],
       [ 941.],
       [ 898.]])}


/tmp/ipykernel_1993396/2486071600.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


IQM
812.909 \textcolor{gray}{[621.17, 898.639]}
Median
813.0 \textcolor{gray}{[702.0, 917.0]}
Mean
714.0 \textcolor{gray}{[585.427, 832.524]}


In [63]:
exp_name = "TD7"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "acrobot-swingup": float(58),
    "ball-in-cup-catch": float(984),
    "cartpole-balance": float(999),
    "cartpole-balance-sparse": float(999),
    "cartpole-swingup": float(869),
    "cartpole-swingup-sparse": float(573),
    "cheetah-run": float(699),
    "finger-spin": float(335),
    "finger-turn-easy": float(912),
    "finger-turn-hard": float(470),
    "fish-swim": float(86),
    "hopper-hop": float(87),
    "hopper-stand": float(670),
    "pendulum-swingup": float(500),
    "quadruped-run": float(645),
    "quadruped-walk": float(949),
    "reacher-easy": float(970),
    "reacher-hard": float(898),
    "walker-run": float(804),
    "walker-stand": float(983),
    "walker-walk": float(977),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

/tmp/ipykernel_1993396/3603924643.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


{'TD7': array([[ 58.],
       [984.],
       [999.],
       [999.],
       [869.],
       [573.],
       [699.],
       [335.],
       [912.],
       [470.],
       [ 86.],
       [ 87.],
       [670.],
       [500.],
       [645.],
       [949.],
       [970.],
       [898.],
       [804.],
       [983.],
       [977.]])}
IQM
771.727 \textcolor{gray}{[570.452, 907.273]}
Median
804.0 \textcolor{gray}{[573.0, 949.0]}
Mean
688.905 \textcolor{gray}{[548.045, 816.239]}


In [64]:
exp_name = "TD-MPC2"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "acrobot-swingup": float(584),
    "ball-in-cup-catch": float(983),
    "cartpole-balance": float(996),
    "cartpole-balance-sparse": float(1000),
    "cartpole-swingup": float(875),
    "cartpole-swingup-sparse": float(845),
    "cheetah-run": float(914),
    "finger-spin": float(986),
    "finger-turn-easy": float(979),
    "finger-turn-hard": float(947),
    "fish-swim": float(659),
    "hopper-hop": float(425),
    "hopper-stand": float(952),
    "pendulum-swingup": float(846),
    "quadruped-run": float(942),
    "quadruped-walk": float(963),
    "reacher-easy": float(983),
    "reacher-hard": float(960),
    "walker-run": float(854),
    "walker-stand": float(991),
    "walker-walk": float(981),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

{'TD-MPC2': array([[ 584.],
       [ 983.],
       [ 996.],
       [1000.],
       [ 875.],
       [ 845.],
       [ 914.],
       [ 986.],
       [ 979.],
       [ 947.],
       [ 659.],
       [ 425.],
       [ 952.],
       [ 846.],
       [ 942.],
       [ 963.],
       [ 983.],
       [ 960.],
       [ 854.],
       [ 991.],
       [ 981.]])}


/tmp/ipykernel_1993396/1275844210.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


IQM
940.909 \textcolor{gray}{[880.543, 972.727]}
Median
952.0 \textcolor{gray}{[875.0, 981.0]}
Mean
888.81 \textcolor{gray}{[820.0, 945.287]}


In [65]:
exp_name = "MR.Q"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "acrobot-swingup": float(567),
    "ball-in-cup-catch": float(981),
    "cartpole-balance": float(999),
    "cartpole-balance-sparse": float(1000),
    "cartpole-swingup": float(866),
    "cartpole-swingup-sparse": float(798),
    "cheetah-run": float(877),
    "finger-spin": float(937),
    "finger-turn-easy": float(953),
    "finger-turn-hard": float(950),
    "fish-swim": float(792),
    "hopper-hop": float(251),
    "hopper-stand": float(951),
    "pendulum-swingup": float(748),
    "quadruped-run": float(947),
    "quadruped-walk": float(963),
    "reacher-easy": float(983),
    "reacher-hard": float(977),
    "walker-run": float(793),
    "walker-stand": float(988),
    "walker-walk": float(978),
}
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

{'MR.Q': array([[ 567.],
       [ 981.],
       [ 999.],
       [1000.],
       [ 866.],
       [ 798.],
       [ 877.],
       [ 937.],
       [ 953.],
       [ 950.],
       [ 792.],
       [ 251.],
       [ 951.],
       [ 748.],
       [ 947.],
       [ 963.],
       [ 983.],
       [ 977.],
       [ 793.],
       [ 988.],
       [ 978.]])}


/tmp/ipykernel_1993396/4141835717.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


IQM
927.0 \textcolor{gray}{[857.361, 966.909]}
Median
950.0 \textcolor{gray}{[866.0, 977.0]}
Mean
871.381 \textcolor{gray}{[787.331, 936.857]}


In [66]:
exp_name = "PPO"
env_step = DMC_STEPS
metric = "avg_return"

your_dict = {
    "acrobot-swingup": float(39),
    "ball-in-cup-catch": float(769),
    "cartpole-balance": float(999),
    "cartpole-balance-sparse": float(1000),
    "cartpole-swingup": float(776),
    "cartpole-swingup-sparse": float(391),
    "cheetah-run": float(269),
    "finger-spin": float(459),
    "finger-turn-easy": float(182),
    "finger-turn-hard": float(58),
    "fish-swim": float(103),
    "hopper-hop": float(10),
    "hopper-stand": float(128),
    "pendulum-swingup": float(115),
    "quadruped-run": float(144),
    "quadruped-walk": float(122),
    "reacher-easy": float(367),
    "reacher-hard": float(125),
    "walker-run": float(97),
    "walker-stand": float(431),
    "walker-walk": float(283),
}
num_env = len(your_dict.keys())
df = pd.DataFrame(columns=["exp_name", "env_name", "env_step", "metric", "value", "seed"])
for k, v in your_dict.items():
    df = pd.concat([df, 
                    pd.DataFrame.from_dict([{"exp_name": exp_name, "env_name": k, "env_step": env_step, "metric": metric, "value": v, "seed": 0}])], ignore_index=True)

df = generate_metric_matrix_dict(
    df, 
    env_step=env_step, 
    metric_type=metric,
)
print(df)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    df, aggregate_func, reps=10000
)
# print(aggregate_scores)
for i, agg in enumerate(["IQM", "Median", "Mean"]):
    data = {"\\textbf{Task}": agg}
    mean = aggregate_scores[exp_name][i] 
    low_CI = aggregate_score_cis[exp_name][0][i] 
    high_CI = aggregate_score_cis[exp_name][1][i]
    
    if metric == "avg_success": 
        mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
    else:
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)

    print(agg)
    print(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    

{'PPO': array([[  39.],
       [ 769.],
       [ 999.],
       [1000.],
       [ 776.],
       [ 391.],
       [ 269.],
       [ 459.],
       [ 182.],
       [  58.],
       [ 103.],
       [  10.],
       [ 128.],
       [ 115.],
       [ 144.],
       [ 122.],
       [ 367.],
       [ 125.],
       [  97.],
       [ 431.],
       [ 283.]])}


/tmp/ipykernel_1993396/1931074259.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,


IQM
232.455 \textcolor{gray}{[133.082, 411.093]}
Median
182.0 \textcolor{gray}{[122.0, 391.0]}
Mean
327.0 \textcolor{gray}{[203.807, 461.098]}
